# WGU Capstone Project: Wyoming Weather Prediction System
This Juypter notebook is used to perform the feature engineering, model training, and evalutation of several techniques for predicting Numerical Weather data.
The data is ingested from an InfluxDB bucket that pulls archived weather data from the OpenMeteo API. The data is stored in a bucket named `OpenMeteo` and is ingested hourly.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from weather_utils import fetch_influx_data, engineer_features, write_influx_data

df_raw = fetch_influx_data()
print(f"Data loaded. Shape: {df_raw.shape}")
print(f"Locations: {df_raw['location'].unique()}")
df_raw.head()

Data loaded. Shape: (26280, 19)
Locations: <StringArray>
['dubois', 'lander', 'pinedale']
Length: 3, dtype: str


,time,day_of_week,location,month_name,season,cloud_cover,cloud_cover_high,cloud_cover_low,cloud_cover_mid,hour,is_weekend,month,precipitation,rain,relative_humidity,snow_depth,snowfall,temperature,wind_speed
8208,2025-03-15 03:00:00+00:00,Saturday,dubois,Mar,spring,90.0,67.0,68.0,37.0,3,1,3,0.0,0.0,67.500954,0.82,0.0,-5.8965,12.860871
8209,2025-03-15 04:00:00+00:00,Saturday,dubois,Mar,spring,100.0,100.0,29.0,97.0,4,1,3,0.0,0.0,68.069565,0.82,0.0,-8.1965,12.224107
8210,2025-03-15 05:00:00+00:00,Saturday,dubois,Mar,spring,96.0,95.0,1.0,87.0,5,1,3,0.0,0.0,71.920998,0.82,0.0,-10.1465,11.734564
8211,2025-03-15 06:00:00+00:00,Saturday,dubois,Mar,spring,87.0,80.0,19.0,64.0,6,1,3,0.0,0.0,74.137154,0.82,0.0,-11.0465,12.033103
8212,2025-03-15 07:00:00+00:00,Saturday,dubois,Mar,spring,100.0,98.0,77.0,60.0,7,1,3,0.0,0.0,75.471825,0.82,0.0,-12.1465,12.074766


### Initial Data Investigation and Tests
Identify the date range, shape, and missingness for key fields.

In [2]:
print("Date range:", df_raw["time"].min(), "→", df_raw["time"].max())
print("Rows per location:")
print(df_raw["location"].value_counts())

key_fields = [
    "temperature",
    "precipitation",
    "wind_speed",
    "relative_humidity",
    "cloud_cover",
    "rain",
    "snowfall",
]
available = [col for col in key_fields if col in df_raw.columns]
if available:
    print("\nMissing values by field:")
    print(df_raw[available].isna().mean().sort_values(ascending=False))

Date range: 2025-03-15 03:00:00+00:00 → 2026-03-15 02:00:00+00:00
Rows per location:
location
dubois      8760
lander      8760
pinedale    8760
Name: count, dtype: int64

Missing values by field:
temperature          0.0
precipitation        0.0
wind_speed           0.0
relative_humidity    0.0
cloud_cover          0.0
rain                 0.0
snowfall             0.0
dtype: float64


### Feature Engineering

Prior to training this step will engineer features to capture temporal trends, lagged features, and rolling statistics to aid in model training.
1.  **Temporal Features**: Extract hour, day, month from timestamp.
2.  **Lagged Features**: Use past values (e.g., 1h, 3h, 6h ago) to predict future ones.
3.  **Rolling Statistics**: Moving averages and standard deviations to capture trends.


In [ ]:
df_engineered = engineer_features(df_raw)
print(f"Feature Engineering complete. Shape: {df_engineered.shape}")

### Feature Selection & Correlation
Let's look at how features correlate with our targets. This is merely a snapshot of the features and does not prove causality.

I have also excluded `time` and `location` from the correlation matrix to avoid trivial correlations.


In [ ]:
targets = ['temperature', 'precipitation', 'wind_speed', 'cloud_cover', 'rain', 'snowfall']

corr_matrix = df_engineered.drop(columns=['time', 'location']).corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix[targets].sort_values(by=targets, ascending=False).head(20), annot=True, cmap='coolwarm')
plt.title("Correlation of Top Features with Targets")
plt.show()

The correlation matrix calculated as shown above offers insights into what we may come to expect from this type of numerical modeling.
For example, the temperature-lagged features are of high importance and likely are correlated. Though features like Cloud_Cover, and wind_speed will be challenging to infer/predict based on the current set of features.
The higher precipitation values, as shown for rain_roll_std_6 rain_roll_mean_6, may be of use for predicting Precipitation, or rain likelihood, as it will be challenging to predict rain likelihood without including radar data.

### Model Training
Five models will be used for training, and the best performing model will be used for future predictions
- Random forests handle nonlinear relationships and mixed feature types well
- Multi-output regression predicts multiple targets in a single pass
- Gradient Boosting Machines (GBM) are known for their ability to handle complex interactions and are often used for regression tasks
- Ridge regression is a linear regression model that attempts to minimize the L2 norm of the residuals
- XGBoost is a fast, scalable, and effective gradient boosting library that can handle large datasets and complex interactions
- LightGBM is a fast, scalable gradient boosting framework that provides a more efficient and flexible alternative to XGBoost

**Important:** We split **chronologically** (shuffle=False) to avoid future data leaking into the training set.


In [ ]:
# 1. Define Features and Targets
drop_cols = ['time', 'location'] + targets
X = df_engineered.drop(columns=drop_cols)
y = df_engineered[targets]

# 2. Encode categorical 'location' if we want to use it as a feature
# We use pd.get_dummies for now, ensuring all expected locations are present
X = pd.concat([X, pd.get_dummies(df_engineered['location'], prefix='loc')], axis=1)

# Ensure only numeric data is in X
X = X.select_dtypes(include=[np.number, bool])

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. Model Definitions
models = {
    "Random Forest": MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    "Linear Regression": MultiOutputRegressor(LinearRegression()),
    "Ridge Regression": MultiOutputRegressor(Ridge(alpha=1.0)),
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)),
    "LightGBM": MultiOutputRegressor(LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
}

# 5. Fit Models
for name, model in models.items():
    print(f"Training {name} model...")
    model.fit(X_train, y_train)
    print(f"Training {name} complete.")

### Model Evaluation
With the models successfully trained, the next step is to evaluate their performance, in this case we can use the Root Mean Squared Error (RMSE) and R² Score for each target.

In [ ]:
from collections import namedtuple
import pandas as pd

Score = namedtuple('Score', 'model target rmse r2 stdev')
all_scores = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    for i, target in enumerate(targets):
        rmse = root_mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
        stdev = np.std(y_test.iloc[:, i])
        all_scores.append(Score(name, target, rmse, r2, stdev))

# Create a DataFrame for easier comparison
scores_df = pd.DataFrame(all_scores)

# Display scores
for name in models.keys():
    print(f"\n--- {name} Performance ---")
    model_scores = scores_df[scores_df['model'] == name]
    for _, score in model_scores.iterrows():
        print(f"rmse={score.rmse:.2f}, r2={score.r2:.2f}, stdev={score.stdev:.2f} | {score.target}")

# Summary Comparison for a key target (e.g., Temperature)
print("\n--- Model Comparison (Temperature) ---")
temp_comparison = scores_df[scores_df['target'] == 'temperature'].sort_values('rmse')
print(temp_comparison[['model', 'rmse', 'r2']])

In [ ]:
# Feature Importance for the Random Forest Model (Aggregated)
rf_model = models["Random Forest"]
importances = np.mean([est.feature_importances_ for est in rf_model.estimators_], axis=0)
feature_importance_df = pd.DataFrame({'feature': X.columns, 'importance': importances}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance_df.head(15))
plt.title("Top 15 Most Useful Features (Random Forest Aggregated Importance)")
plt.show()

In [ ]:
# Model Performance Comparison Plot (RMSE)
plt.figure(figsize=(12, 6))
sns.barplot(x='model', y='rmse', hue='target', data=scores_df[scores_df['target'].isin(['temperature', 'wind_speed', 'cloud_cover'])])
plt.title("Model Comparison: RMSE for Key Targets")
plt.ylabel("RMSE (Lower is better)")
plt.show()

The results shown above represent the model's relative performance across different targets. The RMSE values indicate the average prediction error for each model and target combination. Lower RMSE values suggest better model performance. The plot provides a visual comparison of how well each model predicts temperature, wind speed, and cloud cover, which are key weather variables for our analysis, considering the previous feature important output.

### Generating a 24-hour Nowcast (Future Predictions)
This section shows a **simple recursive forecast** for the next 24 hours using the trained model:

- We start from the **latest available observations** in `df_engineered`.
- For **exogenous variables** we don’t forecast (e.g., `relative_humidity`), we carry the **last observed value** forward.
- For **targets** (`temperature`, `precipitation`, etc.), we **predict one hour ahead**, then feed that prediction back in to build the next hour’s lag/rolling features.

This is a baseline approach for demonstration. For production forecasting, consider using a model that explicitly supports multi-step forecasting or supplying external forecasts for exogenous inputs.


In [ ]:
from main import get_global_latest_timestamp

features_to_lag = [
    "temperature",
    "precipitation",
    "wind_speed",
    "relative_humidity",
    "cloud_cover",
    "rain",
    "snowfall",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
]
lags = [1, 3, 6, 24]
windows = [6, 24]


def build_future_features(df_history: pd.DataFrame, model, X_columns, targets, horizon: int = 24) -> pd.DataFrame:
    future_rows = []
    locations = df_history["location"].unique()

    # Identify exogenous columns to carry forward
    # These are columns that are NOT targets, NOT time/location, AND NOT engineered features
    engineered_suffixes = ("_lag_1", "_lag_3", "_lag_6", "_lag_24", "_roll_mean_6", "_roll_std_6", "_roll_mean_24", "_roll_std_24")
    temporal_cols = ("hour", "day_of_year", "month", "hour_sin", "hour_cos")

    # Base columns are those that exist in df_history before engineering,
    # minus targets, time, and location.
    exogenous_cols = [
        col for col in df_history.columns
        if col not in targets
        and col not in ["time", "location"]
        and not col.endswith(engineered_suffixes)
        and col not in temporal_cols
    ]

    for loc in locations:
        # Get history for this location and sort by time
        hist = df_history[df_history["location"] == loc].sort_values("time").copy()

        for step in range(1, horizon + 1):
            # Calculate next timestamp
            future_time = hist["time"].iloc[-1] + pd.Timedelta(hours=1)

            # Create a new row with time and location
            row = {"time": future_time, "location": loc}

            # Carry forward exogenous variables from the last row
            for col in exogenous_cols:
                row[col] = hist[col].iloc[-1]

            # Add placeholders for targets (required for engineer_features to work correctly)
            for target in targets:
                row[target] = np.nan

            # Append placeholder row to history
            hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)

            # Re-engineer features for the history to get lags/rolling for the new row
            # Note: engineer_features handles temporal encoding automatically
            hist_engineered = engineer_features(hist)

            # Get the engineered features for the latest row
            latest_row_engineered = hist_engineered.iloc[-1:]

            # Prepare feature matrix X for prediction
            # 1. Drop non-feature columns
            step_X = latest_row_engineered.drop(columns=["time", "location"] + targets, errors="ignore")

            # 2. Add location dummy columns (one-hot encoding)
            loc_dummies = pd.get_dummies(latest_row_engineered['location'], prefix='loc')
            step_X = pd.concat([step_X.reset_index(drop=True), loc_dummies], axis=1)

            # 3. Ensure all expected feature columns are present and in the correct order
            step_X = step_X.reindex(columns=X_columns, fill_value=0)

            # Predict all targets for this time step
            step_pred = model.predict(step_X)[0]

            # Update history and the result row with predicted values
            for idx, target in enumerate(targets):
                val = float(step_pred[idx])
                hist.loc[hist.index[-1], target] = val
                row[target] = val

            future_rows.append(row)

    # Combine all predicted rows into a final DataFrame
    df_predictions = pd.DataFrame(future_rows)

    # Store predictions in InfluxDB
    store_cols = ["time", "location"] + targets
    df_store = df_predictions[store_cols]
    write_influx_data(df_store, bucket="OpenMeteo", measurement="hourly_weather_predictions")

    return df_predictions


latest_timestamp = get_global_latest_timestamp()
print(f"Latest timestamp in InfluxDB: {latest_timestamp}")

# Use the best performing model for future predictions (e.g., Random Forest or XGBoost)
# For this scaffold, we'll continue with Random Forest
best_model = models["Random Forest"]
future_predictions = build_future_features(df_engineered, best_model, X.columns, targets, horizon=24)
future_predictions.head()

In [ ]:
# 2. Evaluate performance for different forecast horizons (h)
horizons = [1, 2, 3, 6, 12, 24]
results = []

for h in horizons:
    # Target is the temperature 'h' hours in the future
    # We want to predict temp[t+h] using data available at time t
    y = df_engineered['temperature'].shift(-h).dropna()
    X = df_engineered.drop(columns=['temperature']).iloc[:-h]

    # Encode categorical 'location' and ensure only numeric features
    X = pd.concat([X.drop(columns=['location', 'time']), pd.get_dummies(X['location'], prefix='loc')], axis=1)
    X = X.select_dtypes(include=[np.number, bool])

    # Split into train and test
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Train Random Forest
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predict and calculate error
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    results.append({'Horizon': h, 'MAE': mae, 'RMSE': rmse})

# Convert to DataFrame for visualization
results_df = pd.DataFrame(results)
print(results_df)

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(results_df['Horizon'], results_df['RMSE'], marker='o', label='RMSE')
plt.plot(results_df['Horizon'], results_df['MAE'], marker='s', label='MAE')
plt.title('Forecast Error vs. Horizon (Hours)')
plt.xlabel('Horizon (Hours in future)')
plt.ylabel('Error (Celsius)')
plt.grid(True)
plt.legend()
plt.show()